# Module 09 — Neural N-Gram Model (Bengio et al. 2003)

The bigram model (Module 06) only ever looks at **one** previous
character — `P(next | prev)`. Real language depends on more context than
that. The obvious fix is a counting table over *n* previous characters
instead of 1 — but that table has `vocab_size^n` rows: with our 27-character
vocabulary, a context of 3 needs `27^3 = 19,683` rows, almost all of which
would never appear in a small dataset and would need heavy smoothing to
even be usable.

Bengio et al. (2003) proposed the fix that's still the core idea behind
every modern LLM: instead of a lookup table, **embed** each of the *n*
context tokens (Module 08), concatenate their embeddings, and feed the
result through an MLP to predict the next token. The MLP *shares*
parameters across every context, so it can generalize to contexts it never
saw exactly, rather than needing every possible context spelled out.

## 1. Building a fixed-context dataset (same names + train/test split as Module 07)

In [ ]:
import random

import torch
import torch.nn as nn
import torch.nn.functional as F

names = [
    "aether", "lumine", "amber", "kaeya", "lisa", "jean", "barbara", "diluc",
    "noelle", "bennett", "fischl", "sucrose", "chongyun", "klee", "xingqiu",
    "ningguang", "beidou", "xiangling", "xiao", "zhongli", "hutao", "yanfei",
    "rosaria", "albedo", "diona", "mona", "keqing", "qiqi", "venti",
    "tartaglia", "ganyu", "xinyan", "sayu", "kokomi", "kazuha", "ayaka",
    "yoimiya", "sara", "raiden", "aloy", "itto", "gorou", "yaemiko",
    "shinobu", "heizou", "yelan", "tighnari", "nahida", "nilou", "cyno",
    "candace", "layla", "wanderer", "faruzan", "dehya", "mika", "kaveh",
    "baizhu", "kirara", "lynette", "lyney", "freminet", "neuvillette",
    "wriothesley", "charlotte", "furina", "chevreuse", "navia", "chiori",
    "arlecchino", "clorinde", "sigewinne", "emilie", "kachina", "kinich",
    "mualani", "xilonen", "ororon", "chasca", "mavuika", "citlali", "varesa",
    "iansan", "escoffier", "ineffa",
]

vocab = ["."] + sorted(set("".join(names)))
stoi = {ch: i for i, ch in enumerate(vocab)}
vocab_size = len(vocab)

random.seed(42)
shuffled = names[:]
random.shuffle(shuffled)
split = int(0.8 * len(shuffled))
train_names, test_names = shuffled[:split], shuffled[split:]

BLOCK_SIZE = 3  # how many previous characters the model conditions on

def build_dataset(name_list):
    contexts, targets = [], []
    for name in name_list:
        context = [stoi["."]] * BLOCK_SIZE
        for ch in name + ".":
            contexts.append(context)
            targets.append(stoi[ch])
            context = context[1:] + [stoi[ch]]
    return torch.tensor(contexts), torch.tensor(targets)

X_train, Y_train = build_dataset(train_names)
X_test, Y_test = build_dataset(test_names)
print(f"train examples: {X_train.shape[0]}, test examples: {X_test.shape[0]}")
print("example context -> target:", [vocab[i] for i in X_train[0].tolist()], "->", vocab[Y_train[0].item()])

## 2. The model: embed each context position, concatenate, MLP

`vocab_size^n` table entries becomes `vocab_size * embed_dim` embedding
parameters plus one shared MLP — dramatically fewer parameters, and every
context benefits from what the model learned about each *character*
regardless of which exact 3-character window it appeared in.

In [ ]:
class NGramMLP(nn.Module):
    def __init__(self, vocab_size, embed_dim, block_size, hidden_dim):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.hidden = nn.Linear(embed_dim * block_size, hidden_dim)
        self.out = nn.Linear(hidden_dim, vocab_size)

    def forward(self, context_idx):
        emb = self.embed(context_idx)               # (batch, block_size, embed_dim)
        emb = emb.view(emb.shape[0], -1)             # concat along the context dimension
        h = torch.tanh(self.hidden(emb))
        return self.out(h)


torch.manual_seed(42)
model = NGramMLP(vocab_size, embed_dim=8, block_size=BLOCK_SIZE, hidden_dim=32)
n_table_entries = vocab_size ** BLOCK_SIZE
n_model_params = sum(p.numel() for p in model.parameters())
print(f"A counting table over {BLOCK_SIZE}-character contexts would need {n_table_entries} rows.")
print(f"This MLP has {n_model_params} parameters total, and generalizes across contexts.")

## 3. Training, tracking train vs. test loss

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.05, weight_decay=1e-3)

train_losses, test_losses = [], []
for step in range(300):
    logits = model(X_train)
    loss = F.cross_entropy(logits, Y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    with torch.no_grad():
        test_loss = F.cross_entropy(model(X_test), Y_test)
    train_losses.append(loss.item())
    test_losses.append(test_loss.item())

    if step % 50 == 0:
        print(f"step {step:3d}   train loss {loss.item():.4f}   test loss {test_loss.item():.4f}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
plt.plot(train_losses, label="train")
plt.plot(test_losses, label="test")
plt.xlabel("step")
plt.ylabel("loss (nats)")
plt.legend()
plt.title("Neural n-gram: train vs. test loss")
plt.show()

## 4. Comparing perplexity against Module 07's bigram baseline

`cross_entropy` already computes average negative log-likelihood in nats,
so `exp(loss)` is exactly the perplexity from Module 07 — same metric,
directly comparable.

In [ ]:
best_test_loss = min(test_losses)
ngram_test_ppl = torch.exp(torch.tensor(best_test_loss)).item()
bigram_test_ppl = 15.91  # from Module 07, same train/test split and names

print(f"Neural {BLOCK_SIZE}-gram best test perplexity: {ngram_test_ppl:.2f}")
print(f"Module 07 bigram test perplexity:      {bigram_test_ppl:.2f}")
if ngram_test_ppl < bigram_test_ppl:
    print("\nThe neural n-gram model, using more context, generalizes better than the 1-character bigram model.")
else:
    print("\nOn this tiny dataset the extra context didn\'t clearly help - a good reminder that more context needs enough data to back it up, otherwise it just overfits.")

## 5. Sampling from the trained model

In [ ]:
generator = torch.Generator().manual_seed(42)

def sample_name():
    context = [stoi["."]] * BLOCK_SIZE
    out = []
    for _ in range(30):
        logits = model(torch.tensor([context]))
        probs = F.softmax(logits, dim=-1)
        ix = torch.multinomial(probs[0], num_samples=1, generator=generator).item()
        if ix == stoi["."]:
            break
        out.append(vocab[ix])
        context = context[1:] + [ix]
    return "".join(out)


for _ in range(10):
    print(sample_name())

## Recap

- Counting-based n-gram models need `vocab_size^n` table entries — infeasible
  past small `n`. The neural n-gram model instead learns an embedding per
  token plus a shared MLP, so its parameter count grows linearly (not
  exponentially) with context length, and it generalizes to contexts it
  never saw verbatim.
- We reused the exact same evaluation (perplexity on held-out data) from
  Module 07, so the two models are directly comparable.
- What's still missing: this model only sees a **fixed** window of 3
  characters, chosen in advance. A Transformer's attention mechanism (Phase
  3, starting next module) lets the model look back over its *entire*
  context and learn *for itself* which earlier tokens matter most for
  predicting the next one — no fixed window required.